In [1]:
# Load Paris text file
with open("paris.txt", "r", encoding="utf-8") as f:
    paris_text = f.read()

# Split into sentences (simple split; can be improved)
sentences = [s.strip() for s in paris_text.split(".") if s.strip()]

print(f"Loaded {len(sentences)} sentences from paris.txt")
sentences[:5]

Loaded 4 sentences from paris.txt


['paris is capital of france',
 'paris has eiffle tower',
 'paris has 6 rivers flowing',
 'paris has french coffee']

In [2]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")
import numpy as np

sentence_embeddings = model.encode(sentences, convert_to_numpy=True)
sentence_embeddings.shape

def cosine_similarity(query_vec, doc_matrix):
    query_norm = query_vec / np.linalg.norm(query_vec)
    doc_norm = doc_matrix / np.linalg.norm(doc_matrix, axis=1, keepdims=True)
    return np.dot(doc_norm, query_norm)

c:\Users\nkone\GitHub\repos\Langgraphcourse\tenv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7747.59it/s]


In [3]:
def chatbot(query, top_k=3):
    # Embed user query
    query_emb = model.encode([query], convert_to_numpy=True)[0]
    
    # Compute similarity
    scores = cosine_similarity(query_emb, sentence_embeddings)
    #print("All sentence scores:")
    #print(f"Query: {query}")
    #print(len(scores))
    #for i in range(len(scores)):
    #    print(f"Score: {scores[i]:.4f} - Sentence: {sentences[i]}")
    
    # Rank
    top_idx = np.argsort(scores)[::-1][:top_k]
    #print(len(top_idx))
    #print(f"Top {top_k} matching sentences:")
    #for i in top_idx:
    #    print(f"Score: {scores[i]:.4f} - Sentence: {sentences[i]}")
    
    # Return best matching sentences
    responses = [sentences[i] for i in top_idx]
    return responses

In [4]:
print("Paris Chatbot is ready. Type 'exit' to stop.\n")

while True:
    user_input = input("You: ")
    
    if user_input.lower() in ["exit", "quit", "bye"]:
        print("Chatbot: Goodbye!")
        break
    
    answers = chatbot(user_input)
    
    print("\nChatbot:")
    print(" the  matching answer is " + (answers[0] if answers else "No answer found."))
    print("\n\nOther relevant information of paris is below:   ")
    for ans in answers:
        print("-", ans)
    print("\n")

Paris Chatbot is ready. Type 'exit' to stop.


Chatbot:
 the  matching answer is paris has 6 rivers flowing


Other relevant information of paris is below:   
- paris has 6 rivers flowing
- paris is capital of france
- paris has french coffee


Chatbot: Goodbye!
